In [ ]:
# ==========================================
# Confocal Microscopy Fiber Analysis
# ==========================================

import matplotlib
# Force "Agg" backend to prevent crashes
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
from skimage import io, color, filters, morphology, util, measure
from scipy import ndimage
from scipy.stats import ttest_ind, f_oneway
import os
import glob
import datetime
import warnings

# Import statsmodels for Post-Hoc tests
try:
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print("WARNING: 'statsmodels' not installed. Post-hoc tests will be skipped.")

# ==========================================
# 1. SETUP: Define paths and settings
# ==========================================

# Format: "Label for Graph": r"Path_To_Folder"
DATASET_PATHS = {
   # "Control": r'pathname',
   # "Treatment A": r'pathname',
   # "Treatment B": r'pathname',
}

# Analysis Settings
MIN_THICKNESS = 1       # Minimum fiber thickness in pixels (0 for no filtering)
MIN_ASPECT_RATIO = 1.5  # Minimum aspect ratio (length/width) to be counted as a fiber
MIN_AREA = 5           # Minimum area in pixels to remove background noise/speckles

warnings.filterwarnings("ignore")

# ==========================================
# PROCESSING FUNCTIONS
# ==========================================

def analyze_single_group(input_folder, group_label, min_thickness, min_aspect_ratio, min_area):
    """
    Runs Otsu analysis on a folder of images.
    Returns a DataFrame of metrics and saves per-image visualizations.
    """
    # Setup Output
    output_folder = os.path.join(input_folder, 'Results_Otsu')
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Find Images
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.tif', '*.tiff']
    image_files = []
    for ext in extensions:
        image_files.extend(glob.glob(os.path.join(input_folder, ext)))
        image_files.extend(glob.glob(os.path.join(input_folder, ext.upper())))
    image_files = sorted(list(set(image_files)))
    
    if not image_files:
        print(f"[{group_label}] No images found in {input_folder}")
        return pd.DataFrame()

    print(f"[{group_label}] Processing {len(image_files)} files...")

    # Create footprint for cleaning
    footprint = None
    if min_thickness > 0:
        radius = max(1, int(min_thickness // 2))
        footprint = morphology.disk(radius)

    group_results = []

    for i, image_path in enumerate(image_files):
        filename = os.path.basename(image_path)
        base_name = os.path.splitext(filename)[0]
        
        try:
            # Load Image
            raw_img = io.imread(image_path)

            # --- PREPARE FRAMES (Handle Stacks vs Single) ---
            frames_to_process = [] 

            # Case 1: Single Grayscale (2D)
            if raw_img.ndim == 2:
                frames_to_process.append((raw_img, ""))
            # Case 2: Single RGB (3D) -> Grayscale
            elif raw_img.ndim == 3 and raw_img.shape[-1] in [3, 4]:
                gray = color.rgb2gray(raw_img)
                frames_to_process.append((gray, ""))
            # Case 3: Grayscale Stack (3D)
            elif raw_img.ndim == 3:
                for f_idx in range(raw_img.shape[0]):
                    frames_to_process.append((raw_img[f_idx], f"_frame_{f_idx+1:03d}"))
            # Case 4: RGB Stack (4D)
            elif raw_img.ndim == 4:
                for f_idx in range(raw_img.shape[0]):
                    gray = color.rgb2gray(raw_img[f_idx])
                    frames_to_process.append((gray, f"_frame_{f_idx+1:03d}"))

            # --- PROCESS FRAMES ---
            for gray_img, suffix in frames_to_process:
                # Normalize 
                gray_img = gray_img.astype(float)
                if gray_img.max() > 1.0:
                    gray_img /= gray_img.max()

                # --- 1. OTSU THRESHOLDING ---
                try:
                    thresh = filters.threshold_otsu(gray_img)
                    mask = gray_img > thresh
                except ValueError:
                    mask = np.zeros_like(gray_img, dtype=bool)

                # --- 2. CLEANING (Opening & Area Filter) ---
                if footprint is not None:
                    mask = morphology.binary_opening(mask, footprint)
                
                # Remove small speckles
                if min_area > 0:
                    mask = morphology.remove_small_objects(mask, min_size=min_area)

                # --- 3. LABELING & FILTERING BY ASPECT RATIO ---
                initial_label_image = measure.label(mask)
                valid_labels = []
                
                for region in measure.regionprops(initial_label_image):
                    # Prevent division by zero
                    if region.minor_axis_length == 0:
                        continue
                        
                    aspect_ratio = region.major_axis_length / region.minor_axis_length
                    
                    if aspect_ratio >= min_aspect_ratio:
                        valid_labels.append(region.label)
                
                # Create the final mask with only the true fibers
                mask = np.isin(initial_label_image, valid_labels)
                label_image = measure.label(mask)
                num_features = len(valid_labels)

                # Save Labeled Image
                save_name_labeled = f"{base_name}{suffix}_labeled.png"
                image_label_overlay = color.label2rgb(label_image, image=gray_img, bg_label=0, alpha=0.3)
                io.imsave(os.path.join(output_folder, save_name_labeled), util.img_as_ubyte(image_label_overlay))

                # --- 4. METRICS ---
                if np.sum(mask) == 0:
                    widths = [0]
                    sbr = 0
                else:
                    dist = ndimage.distance_transform_edt(mask)
                    skel = morphology.skeletonize(mask)
                    widths = dist[skel] * 2
                    if len(widths) == 0: widths = [0]
                    
                    fg = np.mean(gray_img[mask])
                    bg = np.mean(gray_img[~mask]) if np.sum(~mask) > 0 else 0
                    sbr = fg / bg if bg > 0 else 0

                stats = {
                    "Filename": f"{filename}{suffix}", 
                    "Group": group_label,
                    "SBR": sbr,
                    "Fiber Count": num_features,
                    "Fiber Density (%)": np.sum(mask) / mask.size * 100,
                    "Mean Width (px)": np.mean(widths),
                    "Median Width (px)": np.median(widths)
                }
                group_results.append(stats)

                # --- 5. PLOTTING (Per Image) ---
                fig, axes = plt.subplots(2, 1, figsize=(6, 8))
                axes[0].imshow(mask, cmap='gray')
                axes[0].set_title(f"Final Mask (Count: {num_features})")
                axes[0].axis('off')
                
                axes[1].hist(widths, bins=30, color='blue', alpha=0.7)
                axes[1].set_title("Fiber Width Distribution")
                axes[1].set_xlabel("Width (px)")
                
                save_name_summ = f"{base_name}{suffix}_summary.png"
                plt.tight_layout()
                plt.savefig(os.path.join(output_folder, save_name_summ))
                plt.close(fig) 
                plt.close('all')

        except Exception as e:
            print(f"Error on {filename}: {e}")

    return pd.DataFrame(group_results)

# ==========================================
# MAIN EXECUTION BLOCK
# ==========================================

if __name__ == "__main__":
    print(f"--- Otsu Fiber Analysis Pipeline ---")
    print(f"Filters -> Min Thickness: {MIN_THICKNESS}px | Min Aspect Ratio: {MIN_ASPECT_RATIO} | Min Area: {MIN_AREA}px")
    
    # 1. Collect Data from All Groups
    all_dfs = []
    for label, path in DATASET_PATHS.items():
        df = analyze_single_group(path, label, MIN_THICKNESS, MIN_ASPECT_RATIO, MIN_AREA)
        if not df.empty:
            all_dfs.append(df)
            
    if all_dfs:
        full_df = pd.concat(all_dfs, ignore_index=True)
        groups_found = full_df['Group'].unique()
        num_groups = len(groups_found)
        
        # Define Metrics to Compare
        numeric_cols = ["SBR", "Fiber Count", "Fiber Density (%)", "Mean Width (px)", "Median Width (px)"]
        
        # 2. Statistical Analysis
        stats_list = []
        posthoc_results = []
        
        print(f"\n--- Statistical Comparison ({num_groups} Groups) ---")
        
        if num_groups < 2:
            print("Not enough groups for statistical comparison.")
        else:
            test_name = "T-Test" if num_groups == 2 else "ANOVA"
            print(f"Test Used: {test_name}\n")
            print(f"{'Feature':<20} | {'P-Value':<10} | {'Significance'}")
            print("-" * 55)

            for col in numeric_cols:
                # Prepare data arrays
                group_data = [full_df[full_df['Group'] == g][col] for g in groups_found]
                
                # Check for zero variance
                if any(g.nunique() <= 1 for g in group_data): continue

                # Run Omnibus Test
                if num_groups == 2:
                    stat, p_val = ttest_ind(group_data[0], group_data[1], equal_var=False)
                else:
                    stat, p_val = f_oneway(*group_data)
                
                sig = "*" if p_val < 0.05 else ""
                print(f"{col:<20} | {p_val:<10.5f} | {sig}")
                
                stats_list.append({'Feature': col, 'Test_Type': test_name, 'P_Value': p_val})

                # Post-Hoc (Tukey)
                if num_groups > 2 and p_val < 0.05 and HAS_STATSMODELS:
                    try:
                        tukey = pairwise_tukeyhsd(endog=full_df[col], groups=full_df['Group'], alpha=0.05)
                        tukey_df = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])
                        tukey_df['Feature'] = col
                        posthoc_results.append(tukey_df)
                    except Exception as e:
                        print(f"  [Error running Post-Hoc for {col}: {e}]")

        # 3. Save Combined Excel
        first_path = list(DATASET_PATHS.values())[0]
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join(first_path, f"Otsu_Analysis_Summary_{timestamp}.xlsx")
        
        try:
            with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
                full_df.to_excel(writer, sheet_name='Raw Data', index=False)
                if stats_list:
                    pd.DataFrame(stats_list).to_excel(writer, sheet_name='Summary Statistics', index=False)
                if posthoc_results:
                    final_posthoc = pd.concat(posthoc_results, ignore_index=True)
                    cols = ['Feature'] + [c for c in final_posthoc.columns if c != 'Feature']
                    final_posthoc[cols].to_excel(writer, sheet_name='Post-Hoc Details', index=False)
                    
            print(f"\n Results saved to: {output_path}")
        except Exception as e:
            print(f"\n Could not save Excel file: {e}")

        # 4. Visualization (Boxplots)
        if stats_list:
            plt.figure(figsize=(14, 10))
            for i, feature_name in enumerate(numeric_cols):
                if i >= 6: break
                plt.subplot(2, 3, i + 1)
                
                plot_data = [full_df[full_df['Group'] == g][feature_name] for g in groups_found]
                p_val = next((item['P_Value'] for item in stats_list if item['Feature'] == feature_name), 1.0)
                
                plt.boxplot(plot_data, labels=groups_found)
                plt.title(f"{feature_name}\n(p={p_val:.1e})")
                plt.grid(True, alpha=0.3)
                plt.xticks(rotation=45)

            plt.tight_layout()
            plt.show() # In Jupyter, this displays inline.
    else:
        print("No data processed.")